# Preprocessing

### Load and Convert Fine-tuned Transformer Model to TransformerLens

In [1]:
import os
import torch
os.environ['CUDA_VISIBLE_DEVICES'] = '6'

In [ ]:
from huggingface_hub import login
login(token='YOUR_HF_TOKEN')

In [4]:
from src import load_finetuned_model, load_finetuned_model_lens_from_dir

# base_model_name = "Qwen/Qwen2.5-0.5B"
# fine_tuned_model_path = "models/fine_tuned_model/"
# fine_tuned_model_path = "absa-research/hotel_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20"
# model = load_finetuned_model(base_model_name, fine_tuned_model_path)
fine_tuned_model_path = "outputs/models/2025-05-21 17:43:35.056477_tflens_hotel_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20"
model = load_finetuned_model_lens_from_dir(fine_tuned_model_path)

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
model.to(device)

model.eval()

Moving model to device:  cuda


HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint()
  (blocks): ModuleList(
    (0-23): 24 x TransformerBlock(
      (ln1): RMSNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): RMSNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): GroupedQueryAttention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
        (hook_rot_k): HookPoint()
        (hook_rot_q): HookPoint()
      )
      (mlp): GatedMLP(
        (hook_pre): HookPoint()
        (hook_pre_linear): HookPoint()
        (hook_post): HookPoint()
      )
      (hook_attn_in): HookPoint()
      (hook_q_input): HookPoint()
      (hook_k_input): HookPoint()
      (hook_v_input): HookPoint()
      (hook_mlp_in): HookPoint()
      (hook_att

## Find the data with the correct answer

In [ ]:
import src
import importlib
importlib.reload(src)
importlib.reload(src.model)
from src import filter_correct_data
import pandas as pd

dataset_path = "hotel_dataset/hotel_aste_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20_counterfactual_correctedv2.csv"
filtered_data_path = "hotel_dataset/hotel_aste_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20_counterfactual_filtered_Av3.csv"
test_df = pd.read_csv(dataset_path)

df_filtered = filter_correct_data(model, test_df, 
                                  "original_sentence", "original_triplet", filter_mode="AOS", filter_only_correct=True, 
                                  save_path=filtered_data_path)

## Create EAP Dataset

#### Building the Dataset

In [ ]:
import src
import importlib
importlib.reload(src)
importlib.reload(src.utils)

from src.utils import build_eap_dataset
import pandas as pd


filtered_data = pd.read_csv(
    "hotel_dataset/hotel_aste_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20_counterfactual_filtered_Av3.csv")

eap_df = build_eap_dataset(
    model=model,
    df=filtered_data,
    sentence_col="original_sentence",
    triplet_col="original_triplet",
    corrupted_col="counterfact1_modified",
    corrupted_triplet_col="counterfact_triplet1_modified",
    suffix="[A]",
    filer_same_length_counterfactuals=True
)
eap_df.to_csv("eap_dataset/eap_dataset_aspect_multitokensv3.csv", index=False)

# EAP-IG

In [ ]:
import gc
import torch

gc.collect()

torch.mps.empty_cache()

In [ ]:
import argparse
import ast
import os
from functools import partial
from random import random
from typing import Optional

import pandas as pd
import transformers
from torch.utils.data import Dataset, DataLoader
import torch
from typing_extensions import Tuple, List, Union
import numpy as np

from eap.graph import Graph
from eap.evaluate import evaluate_graph, evaluate_baseline, evaluate_baseline_multitoken, evaluate_graph_multitoken
from eap.attribute import attribute
from src.utils import build_eap_dataset
from src import load_finetuned_model, filter_correct_data
from src.metric import logit_diff

In [ ]:
# load model
base_model_name = "Qwen/Qwen2.5-0.5B"
fine_tuned_model_path = "models/fine_tuned_model/"
model = load_finetuned_model(base_model_name, fine_tuned_model_path)
model.to("mps")
model.cfg.use_split_qkv_input = True
model.cfg.use_attn_result = True
model.cfg.use_hook_mlp_in = True
model.cfg.ungroup_grouped_query_attention = True

In [ ]:
# load dataset
ds = pd.read_csv("eap_dataset/eap_dataset_aspect_multitokens.csv")

In [ ]:
g = Graph.from_model(model)

In [ ]:
baseline = evaluate_baseline_multitoken(
    model,
    df=ds,
    metrics=[logit_diff],
    run_corrupted=False,
    batch_size=4
)
print(f"Original performance is logit_dif={baseline}")

In [ ]:
attribute(
    model=model,
    graph=g,
    dataloader=ds,  # can be Dataset or DataLoader
    metric=partial(logit_diff,loss=False, mean=True),
    method="EAP-IG-inputs",
    ig_steps=5,
    is_absa=True,
    batch_size=5,
    device="mps"
)

In [ ]:
n_edges = g.real_edge_mask.sum().item()  # total 171K edges for qwen2.5-0.5B
for topk in [100, 200, 500, 1000, 2000, 5000, 10000, 20000]:
    g.reset()
    g.apply_topn(topk, True)
    results = evaluate_graph_multitoken(model=model,
                                        graph=g,
                                        df=ds,  # your full dataset DataFrame
                                        metrics=[partial(logit_diff, mean=True, loss=False)],  # or just [logit_diff]
                                        batch_size=4,)

    print(f"with top-k = {topk} ({topk/n_edges:.1%}), the circuit's performance is {results}, faithfulness={results/baseline:.1%}")
    g.to_pt(f'outputs/opinion_circuit_topk-{topk}.pt')

    print(f"included nodes: {g.count_included_nodes()}, included edges: {g.count_included_edges()}")

# Circuit Merging

In [4]:
from src.utils import edge_merging
topk = [100, 200, 500, 1000, 2000, 5000, 10000, 20000, 30000, 40000]
for k in topk:
	graph_paths = [f"outputs/multitokens/v2/aspect_circuit_topk-{k}.pt", f"outputs/multitokens/v2/sentiment_circuit_topk-{k}.pt", f"outputs/multitokens/v2/opinion_circuit_topk-{k}.pt"]

	complete_edges = edge_merging(graph_paths=graph_paths)

	complete_edges.to_csv(f"outputs/multitokens/v2/complete_circuit_topk-{k}.csv", index=None)

	print(k, complete_edges.shape)

100 (103, 3)
200 (291, 3)
500 (783, 3)
1000 (1634, 3)
2000 (3413, 3)
5000 (9292, 3)
10000 (18715, 3)
20000 (36220, 3)
30000 (52869, 3)
40000 (68467, 3)


In [ ]:
import pandas as pd
from eap.graph import Graph

topk = [100, 200, 500, 1000, 2000, 5000, 10000, 20000, 30000, 40000]
for k in topk:
	df = pd.read_csv(f"outputs/multitokens/v2/complete_circuit_topk-{k}.csv")
	circuit_path = f"outputs/multitokens/v2/aspect_circuit_topk-{k}.pt"
	g = Graph.from_pt(circuit_path)
	print(k, g.count_included_edges())
	number_of_edge = g.count_included_edges()
	for i, edge in enumerate(g.edges.values()):
		if edge.in_graph != True:
			if "m" in edge.child.name:
				mask = (df['parent_node'] == edge.parent.name) & (df['child_node'] == edge.child.name)
			else:
				mask = (df['parent_node'] == edge.parent.name) & (df['child_node'] == edge.child.name) & (df['child_type'] == edge.qkv)
			if sum(mask) == 1:
				edge.in_graph = True
				number_of_edge += 1
				print(k, i, number_of_edge, end="\r")
	g.to_pt(f"outputs/multitokens/v2/complete_circuit_topk-{k}.pt")

100 70


In [ ]:
topk = [100, 200, 500, 1000, 2000, 5000, 10000, 20000, 30000, 40000]

for t in topk:
    print(t)
    df = pd.read_csv(f"outputs/multitokens/complete_circuit_topk-{t}.csv")

    a = []
    for row in df.iterrows():
        if "m" not in row[1]["child_node"] and "logits" not in row[1]["child_node"]:
            a.append(row[1]['child_node'] + ' ' + row[1]['child_type'])

    print(len(np.unique(a)))
    print(df.shape)
    print((df.shape[0]/179387)*100)
    print("\n")

# SFT

## SFT with Active Nodes

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "7"

import json
import torch
from torch.utils.data import Dataset
from transformer_lens.train import train
from transformer_lens.train import HookedTransformerTrainConfig
import pandas as pd
import gc
from src.utils import load_model, ABSAAutoRegressiveDataset
from src.utils import apply_active_edge_unfreezing

In [ ]:
csv_path = "outputs/multitokens/v2/complete_circuit_topk-2000.csv"
json_path = "hotel_dataset/hotel_aste_train_augmented_noreasoning.json"
model_name = "Qwen/Qwen2.5-0.5B"
max_len = 128
device = "cuda"

# === Load ABSA Dataset ===
with open(json_path) as f:
    absa_data = json.load(f)

# === Load Model ===
model = load_model(model_name, device=device)

#we can try several number of samples
dataset = ABSAAutoRegressiveDataset(absa_data[:1000], model.tokenizer)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

#comment it if you want to train full model
apply_active_edge_unfreezing(model, csv_path)

model.train()

# === Train Config ===
config = HookedTransformerTrainConfig(
    num_epochs=5,
    batch_size=32,
    lr=1e-5,
    device=device,
    print_every=10,
)

# === Start Training ===
trained_model = train(model, config, dataset)

In [35]:
import pickle

# Test saving model
save_dir = 'outputs/models/test_model.pt'
torch.save(trained_model.state_dict(), save_dir)

# Convert model.cfg to a dictionary
model_cfg_dict = model.cfg.to_dict()
print(type(model_cfg_dict))

# Assume 'my_dict' is your dictionary containing custom objects
with open('outputs/models/my_dict.pkl', 'wb') as f:
    pickle.dump(model_cfg_dict, f, protocol=pickle.HIGHEST_PROTOCOL)

<class 'dict'>


In [ ]:
model.tokenizer.save_pretrained('outputs/models')

In [ ]:
# Load saved tokenizer
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('outputs/models')
tokenizer

In [ ]:
# Test loading model

from transformer_lens import HookedTransformer, HookedTransformerConfig

with open('outputs/models/my_dict.pkl', 'rb') as f:
	new_cfg_dict = pickle.load(f)
new_cfg = HookedTransformerConfig.from_dict(new_cfg_dict)
new_model = HookedTransformer(new_cfg)
new_model.load_state_dict(torch.load(save_dir))

<All keys matched successfully>

### Inference

In [ ]:
trained_model.generate([ "airnya kurang kencang . [A] [O] [S]"], 
               max_new_tokens=50,
               stop_at_eos=True,
               return_type="str")

### Simple Evaluation

In [ ]:
from src import filter_correct_data
import pandas as pd

# Check the accuracy using data existing counterfact data
# For the reference, the result from the finetune model using transformer is 48 out of 112 sample
dataset_path = "hotel_dataset/hotel_aste_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20_counterfactual_correctedv2.csv"
test_df = pd.read_csv(dataset_path)

df_filtered = filter_correct_data(trained_model, test_df, 
                                  "original_sentence", "original_triplet", filter_mode="AOS", filter_only_correct=False,)

### Full Evaluation

In [ ]:
import src
import importlib
importlib.reload(src)
importlib.reload(src.utils)
from src import load_finetuned_model, load_finetuned_model_lens_from_dir

# base_model_name = "Qwen/Qwen2.5-0.5B"
# fine_tuned_model_path = "models/fine_tuned_model/"
# fine_tuned_model_path = "absa-research/hotel_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20"
# model = load_finetuned_model(base_model_name, fine_tuned_model_path)
fine_tuned_model_path = "outputs/models/2025-05-21 17:43:35.056477_tflens_hotel_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20"
model = load_finetuned_model_lens_from_dir(fine_tuned_model_path)

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
model.to(device)

model.eval()

In [21]:
import json
with open('hotel_dataset/hotel_aste_test_augmented.json', 'r') as f:
	test_data = json.load(f)

In [22]:
prompts = [instance['input'] for instance in test_data]
labels = [instance['target'] for instance in test_data]
sentence_ids = [instance['sentence_id'] for instance in test_data]
tasks = [instance['task_elements'] for instance in test_data]

In [ ]:
from tqdm import tqdm

# Assuming 'model' is a Hugging Face Transformers model and you have a tokenizer

# Your existing data loading
prompts = [instance['input'] for instance in test_data]
labels = [instance['target'] for instance in test_data]
sentence_ids = [instance['sentence_id'] for instance in test_data]
tasks = [instance['task_elements'] for instance in test_data]

# --- Batching Modification ---
batch_size = 5 # You can adjust this based on your GPU memory
outputs = []

for i in tqdm(range(0, len(prompts), batch_size), desc="Generating outputs"):
	batch_prompts = prompts[i:i + batch_size]

	# Generate outputs for the batch
	batch_outputs_text = model.generate(
		input=batch_prompts,
		max_new_tokens=128,
		stop_at_eos=True,
		do_sample=False,
		return_type="str",
		verbose=False,
	)

	if isinstance(batch_outputs_text, str):
		batch_outputs_text = [batch_outputs_text]

	outputs.extend(batch_outputs_text)
	# if i > 14:
	# 	break
# Now 'outputs' contains all the generated strings
len(outputs)


Generating outputs:   0%|          | 3/1000 [00:05<28:18,  1.70s/it]


20

In [ ]:
import src
import importlib
importlib.reload(src)
importlib.reload(src.utils)
from src.utils import postprocess_absa_outputs, calculate_metrics

grouped_per_task = postprocess_absa_outputs(outputs[:20], labels[:20], sentence_ids[:20], tasks[:20])
result_metrics = {}
for task, v in grouped_per_task.items():
	predictions = v["predictions"]
	targets = v["targets"]
	result_metrics.update(
		calculate_metrics(predictions, targets, task)
)
result_metrics

In [ ]:
for key, value in result_metrics.items():
	print(f"{key}: {value * 100}")
new_result_metrics = {key: value * 100 for key, value in result_metrics.items()}

In [84]:
with open(f'outputs/evals/batch_size_1/2025-05-21 17:43:35.056477_tflens_hotel_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20/evaluation_results.json', 'w') as f:
	json.dump(new_result_metrics, f, indent=4)

In [85]:
with open(f'outputs/evals/batch_size_5/2025-05-21 17:43:35.056477_tflens_hotel_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20/inference_results.json', 'w') as f:
	json.dump(outputs, f, indent=4)